#cs224hw3

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp -r /content/drive/MyDrive/'Colab Notebooks'/.ssh /root/

In [ ]:
!git clone https://github.com/dougc333/cs224r.git

Cloning into 'cs224r'...
remote: Enumerating objects: 1633, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 1633 (delta 11), reused 23 (delta 7), pack-reused 1595 (from 2)
Receiving objects: 100% (1633/1633), 441.93 MiB | 31.49 MiB/s, done.
Resolving deltas: 100% (523/523), done.
Updating files: 100% (661/661), done.


In [ ]:
%cd cs224r
!pip install -r requirements.txt

/content/cs224r
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 50.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.8/398.8 kB 31.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.5/374.5 kB 30.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.3/216.3 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.2/750.2 kB 47.9 MB/

In [3]:
!pip install gymnasium-robotics[mujoco-py]

In [4]:
from datetime import datetime
from  pathlib import Path

RUN_NAME = f"dqn_pointmaze_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
LOG_DIR = Path("runs") / RUN_NAME
CKPT_DIR = Path("checkpoints") / RUN_NAME
VIDEO_DIR = Path("videos") / RUN_NAME
print(LOG_DIR)

runs/dqn_pointmaze_20260416_175525


In [5]:
# import os
# os.makedirs(LOG_DIR)


%load tensorboard --logs /content/runs/

ValueError: 'tensorboard --logs /content/runs/' was not found in history, as a file, url, nor in the user namespace.

In [ ]:
class DiscretePointMazeWrapper:
    def __init__(self, env_id=ENV_ID, render_mode=None):
        self.env = gym.make(env_id, render_mode=render_mode)
        self.num_actions = len(ACTIONS)

        obs, info = self.env.reset(seed=SEED)
        self.last_raw_obs = obs
        flat = flatten_obs(obs)
        self.obs_dim = flat.shape[0]

    def reset(self, seed=None):
        obs, info = self.env.reset(seed=seed)
        self.last_raw_obs = obs
        return flatten_obs(obs), info

    def step(self, action_idx: int):
        action = np.clip(ACTIONS[action_idx], -1.0, 1.0).astype(np.float32)
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.last_raw_obs = obs
        done = terminated or truncated
        return flatten_obs(obs), float(reward), done, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


eval_env = DiscretePointMazeWrapper(env_id=ENV_ID, render_mode="rgb_array")
obs, info = eval_env.reset(seed=123)

for t in range(10):
    action = agent.select_action(obs, greedy=True)
    next_obs, reward, done, info = eval_env.step(action)

    print(
        "t =", t,
        "action_idx =", action,
        "action =", np.clip(ACTIONS[action], -1.0, 1.0),
        "achieved_goal =", eval_env.last_raw_obs["achieved_goal"],
        "reward =", reward,
    )

    obs = next_obs
    if done:
        break

In [6]:
import os
import json
import torch
from pathlib import Path

def configure_mujoco_backend():
    # Respect an explicit user setting if already present
    if os.environ.get("MUJOCO_GL"):
        print(f"MUJOCO_GL already set to {os.environ['MUJOCO_GL']}")
        return

    has_gpu = torch.cuda.is_available()

    if has_gpu:
        icd_dir = Path("/usr/share/glvnd/egl_vendor.d")
        icd_dir.mkdir(parents=True, exist_ok=True)

        nvidia_icd_config_path = icd_dir / "10_nvidia.json"
        if not nvidia_icd_config_path.exists():
            nvidia_icd_config_path.write_text(json.dumps({
                "file_format_version": "1.0.0",
                "ICD": {
                    "library_path": "libEGL_nvidia.so.0"
                }
            }, indent=2))

        os.environ["MUJOCO_GL"] = "egl"
        print("GPU detected. Using MuJoCo EGL backend.")
    else:
        os.environ["MUJOCO_GL"] = "osmesa"
        print("No GPU detected. Using MuJoCo OSMesa backend.")

configure_mujoco_backend()

import gymnasium as gym
import gymnasium_robotics

gym.register_envs(gymnasium_robotics)

GPU detected. Using MuJoCo EGL backend.


In [7]:
import os
import json
import random
from datetime import datetime
from pathlib import Path
from collections import deque, namedtuple

import numpy as np
import imageio.v2 as imageio

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

import gymnasium as gym
import gymnasium_robotics

gym.register_envs(gymnasium_robotics)


# =========================================================
# Config
# =========================================================

SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENV_ID = "PointMaze_UMazeDense-v3"

GAMMA = 0.99
LR = 1e-3
BATCH_SIZE = 64
BUFFER_CAPACITY = 100_000
MIN_REPLAY_SIZE = 2_000
TARGET_UPDATE_EVERY = 500
TRAIN_EVERY = 1

EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY_STEPS = 30_000

NUM_EPISODES = 400
LOG_EVERY = 10
MAX_STEPS_PER_EPISODE = 300

CHECKPOINT_EVERY = 50




# =========================================================
# Discrete action set
# =========================================================

ACTIONS = np.array([
    [0.0, 0.0],
    [0.0, -1.0],
    [0.0,  1.0],
    [-1.0, 0.0],
    [1.0,  0.0],
    [-1.0, -1.0],
    [-1.0,  1.0],
    [1.0, -1.0],
    [1.0,  1.0],
], dtype=np.float32)


# =========================================================
# Utils
# =========================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def flatten_obs(obs: dict) -> np.ndarray:
    return np.concatenate(
        [obs["observation"], obs["desired_goal"]],
        axis=0
    ).astype(np.float32)


def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


set_seed(SEED)


# =========================================================
# Env wrapper
# =========================================================

class DiscretePointMazeWrapper:
    def __init__(self, env_id=ENV_ID, render_mode=None):
        self.env = gym.make(env_id, render_mode=render_mode)
        self.num_actions = len(ACTIONS)

        obs, info = self.env.reset(seed=SEED)
        flat = flatten_obs(obs)
        self.obs_dim = flat.shape[0]

    def reset(self, seed=None):
        obs, info = self.env.reset(seed=seed)
        return flatten_obs(obs), info

    def step(self, action_idx: int):
        action = ACTIONS[action_idx]
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return flatten_obs(obs), float(reward), done, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


# =========================================================
# Replay Buffer
# =========================================================

Transition = namedtuple("Transition", ["obs", "action", "reward", "next_obs", "done"])


class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def add(self, obs, action, reward, next_obs, done):
        self.buffer.append(Transition(obs, action, reward, next_obs, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        obs = np.array([t.obs for t in batch], dtype=np.float32)
        actions = np.array([t.action for t in batch], dtype=np.int64)
        rewards = np.array([t.reward for t in batch], dtype=np.float32)
        next_obs = np.array([t.next_obs for t in batch], dtype=np.float32)
        dones = np.array([t.done for t in batch], dtype=np.float32)
        return obs, actions, rewards, next_obs, dones

    def __len__(self):
        return len(self.buffer)

    def state_dict(self):
        return {"buffer": list(self.buffer)}

    def load_state_dict(self, state):
        self.buffer = deque(state["buffer"], maxlen=self.buffer.maxlen)


# =========================================================
# Q network
# =========================================================

class QNetwork(nn.Module):
    def __init__(self, obs_dim: int, num_actions: int, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs)


# =========================================================
# Agent
# =========================================================

class DQNAgent:
    def __init__(self, obs_dim: int, num_actions: int):
        self.num_actions = num_actions

        self.q_net = QNetwork(obs_dim, num_actions).to(DEVICE)
        self.target_net = QNetwork(obs_dim, num_actions).to(DEVICE)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=LR)
        self.replay_buffer = ReplayBuffer(BUFFER_CAPACITY)

        self.total_steps = 0
        self.num_updates = 0

    def epsilon(self) -> float:
        frac = min(1.0, self.total_steps / EPS_DECAY_STEPS)
        return EPS_START + frac * (EPS_END - EPS_START)

    def select_action(self, obs: np.ndarray, greedy: bool = False) -> int:
        if (not greedy) and random.random() < self.epsilon():
            return random.randrange(self.num_actions)

        obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            q_values = self.q_net(obs_t)
            return int(q_values.argmax(dim=1).item())

    def train_step(self):
        if len(self.replay_buffer) < max(MIN_REPLAY_SIZE, BATCH_SIZE):
            return None

        obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(BATCH_SIZE)

        obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
        actions_t = torch.tensor(actions, dtype=torch.long, device=DEVICE)
        rewards_t = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)
        next_obs_t = torch.tensor(next_obs, dtype=torch.float32, device=DEVICE)
        dones_t = torch.tensor(dones, dtype=torch.float32, device=DEVICE)

        q_values = self.q_net(obs_t)
        q_pred = torch.gather(q_values, 1, actions_t.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q_values = self.target_net(next_obs_t)
            next_q_max = next_q_values.max(dim=1).values
            target = rewards_t + GAMMA * (1.0 - dones_t) * next_q_max

        loss = F.smooth_l1_loss(q_pred, target)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        self.num_updates += 1
        if self.num_updates % TARGET_UPDATE_EVERY == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

        td_abs = (q_pred.detach() - target).abs().mean().item()
        q_mean = q_pred.detach().mean().item()
        target_mean = target.detach().mean().item()

        return {
            "loss": float(loss.item()),
            "td_abs": float(td_abs),
            "q_mean": float(q_mean),
            "target_mean": float(target_mean),
        }

    def state_dict(self):
        return {
            "q_net": self.q_net.state_dict(),
            "target_net": self.target_net.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "total_steps": self.total_steps,
            "num_updates": self.num_updates,
            "replay_buffer": self.replay_buffer.state_dict(),
        }

    def load_state_dict(self, state):
        self.q_net.load_state_dict(state["q_net"])
        self.target_net.load_state_dict(state["target_net"])
        self.optimizer.load_state_dict(state["optimizer"])
        self.total_steps = state["total_steps"]
        self.num_updates = state["num_updates"]
        self.replay_buffer.load_state_dict(state["replay_buffer"])


# =========================================================
# Eval + video
# =========================================================

def evaluate(agent: DQNAgent, env: DiscretePointMazeWrapper, num_episodes: int = 10):
    returns = []
    lengths = []

    for ep in range(num_episodes):
        obs, info = env.reset(seed=SEED + 1000 + ep)
        ep_ret = 0.0
        ep_len = 0

        for _ in range(MAX_STEPS_PER_EPISODE):
            action = agent.select_action(obs, greedy=True)
            next_obs, reward, done, info = env.step(action)
            ep_ret += reward
            ep_len += 1
            obs = next_obs
            if done:
                break

        returns.append(ep_ret)
        lengths.append(ep_len)

    return {
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "max_return": float(np.max(returns)),
        "min_return": float(np.min(returns)),
        "mean_length": float(np.mean(lengths)),
    }


def record_video(agent: DQNAgent, env: DiscretePointMazeWrapper, filename="eval.mp4",
                 max_steps=300, fps=20, seed=123):
    frames = []
    obs, info = env.reset(seed=seed)

    frame = env.render()
    if frame is not None:
        frames.append(frame)

    ep_return = 0.0
    for _ in range(max_steps):
        action = agent.select_action(obs, greedy=True)
        obs, reward, done, info = env.step(action)
        ep_return += reward

        frame = env.render()
        if frame is not None:
            frames.append(frame)

        if done:
            break

    out_path = Path(filename)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(out_path, frames, fps=fps)
    print(f"Saved video to {out_path.resolve()} | return={ep_return:.2f}")


# =========================================================
# Checkpoint helpers
# =========================================================

def save_checkpoint(path: Path, agent: DQNAgent, episode: int, best_eval: float, config: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "episode": episode,
        "best_eval": best_eval,
        "config": config,
        "agent": agent.state_dict(),
    }, path)


# =========================================================
# Main
# =========================================================

def main():
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    VIDEO_DIR.mkdir(parents=True, exist_ok=True)

    config = {
        "seed": SEED,
        "env_id": ENV_ID,
        "gamma": GAMMA,
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "buffer_capacity": BUFFER_CAPACITY,
        "min_replay_size": MIN_REPLAY_SIZE,
        "target_update_every": TARGET_UPDATE_EVERY,
        "train_every": TRAIN_EVERY,
        "eps_start": EPS_START,
        "eps_end": EPS_END,
        "eps_decay_steps": EPS_DECAY_STEPS,
        "num_episodes": NUM_EPISODES,
        "max_steps_per_episode": MAX_STEPS_PER_EPISODE,
        "checkpoint_every": CHECKPOINT_EVERY,
        "actions": ACTIONS.tolist(),
        "device": str(DEVICE),
    }
    save_json(LOG_DIR / "config.json", config)

    writer = SummaryWriter(log_dir=str(LOG_DIR))

    train_env = DiscretePointMazeWrapper(env_id=ENV_ID, render_mode=None)
    eval_env = DiscretePointMazeWrapper(env_id=ENV_ID, render_mode="rgb_array")

    agent = DQNAgent(obs_dim=train_env.obs_dim, num_actions=train_env.num_actions)

    train_returns = deque(maxlen=100)
    best_eval_mean = -float("inf")

    for episode in range(1, NUM_EPISODES + 1):
        obs, info = train_env.reset(seed=SEED + episode)
        ep_return = 0.0
        ep_losses = []
        ep_td_abs = []
        ep_q_mean = []
        ep_target_mean = []

        for _ in range(MAX_STEPS_PER_EPISODE):
            agent.total_steps += 1

            action = agent.select_action(obs, greedy=False)
            next_obs, reward, done, info = train_env.step(action)

            agent.replay_buffer.add(obs, action, reward, next_obs, done)

            if agent.total_steps % TRAIN_EVERY == 0:
                stats = agent.train_step()
                if stats is not None:
                    ep_losses.append(stats["loss"])
                    ep_td_abs.append(stats["td_abs"])
                    ep_q_mean.append(stats["q_mean"])
                    ep_target_mean.append(stats["target_mean"])

                    writer.add_scalar("train/loss_step", stats["loss"], agent.total_steps)
                    writer.add_scalar("train/td_abs_step", stats["td_abs"], agent.total_steps)
                    writer.add_scalar("train/q_mean_step", stats["q_mean"], agent.total_steps)
                    writer.add_scalar("train/target_mean_step", stats["target_mean"], agent.total_steps)

            obs = next_obs
            ep_return += reward

            if done:
                break

        train_returns.append(ep_return)

        writer.add_scalar("train/episode_return", ep_return, episode)
        writer.add_scalar("train/episode_return_avg100", np.mean(train_returns), episode)
        writer.add_scalar("train/epsilon", agent.epsilon(), episode)
        writer.add_scalar("train/buffer_size", len(agent.replay_buffer), episode)

        if ep_losses:
            writer.add_scalar("train/loss_episode_mean", np.mean(ep_losses), episode)
            writer.add_scalar("train/td_abs_episode_mean", np.mean(ep_td_abs), episode)
            writer.add_scalar("train/q_mean_episode_mean", np.mean(ep_q_mean), episode)
            writer.add_scalar("train/target_mean_episode_mean", np.mean(ep_target_mean), episode)

        if episode % LOG_EVERY == 0:
            eval_stats = evaluate(agent, eval_env, num_episodes=10)

            writer.add_scalar("eval/mean_return", eval_stats["mean_return"], episode)
            writer.add_scalar("eval/std_return", eval_stats["std_return"], episode)
            writer.add_scalar("eval/max_return", eval_stats["max_return"], episode)
            writer.add_scalar("eval/min_return", eval_stats["min_return"], episode)
            writer.add_scalar("eval/mean_length", eval_stats["mean_length"], episode)

            avg_loss = float(np.mean(ep_losses)) if ep_losses else float("nan")
            print(
                f"Episode {episode:4d} | "
                f"TrainRet {ep_return:8.2f} | "
                f"TrainAvg100 {np.mean(train_returns):8.2f} | "
                f"EvalMean {eval_stats['mean_return']:8.2f} | "
                f"EvalMax {eval_stats['max_return']:8.2f} | "
                f"Eps {agent.epsilon():.3f} | "
                f"Buf {len(agent.replay_buffer):6d} | "
                f"Loss {avg_loss:.4f}"
            )

            if eval_stats["mean_return"] > best_eval_mean:
                best_eval_mean = eval_stats["mean_return"]
                save_checkpoint(
                    CKPT_DIR / "best.pt",
                    agent=agent,
                    episode=episode,
                    best_eval=best_eval_mean,
                    config=config,
                )
                print(f"Saved best checkpoint at episode {episode}, eval mean {best_eval_mean:.2f}")

        if episode % CHECKPOINT_EVERY == 0:
            save_checkpoint(
                CKPT_DIR / f"episode_{episode}.pt",
                agent=agent,
                episode=episode,
                best_eval=best_eval_mean,
                config=config,
            )

    save_checkpoint(
        CKPT_DIR / "final.pt",
        agent=agent,
        episode=NUM_EPISODES,
        best_eval=best_eval_mean,
        config=config,
    )
    torch.save(agent.q_net.state_dict(), CKPT_DIR / "q_net_only.pt")

    record_video(
        agent,
        eval_env,
        filename=VIDEO_DIR / "eval_final.mp4",
        max_steps=MAX_STEPS_PER_EPISODE,
        fps=20,
        seed=SEED + 999,
    )

    writer.flush()
    writer.close()
    train_env.close()
    eval_env.close()


if __name__ == "__main__":
    main()

Episode   10 | TrainRet    55.96 | TrainAvg100    62.31 | EvalMean    63.54 | EvalMax   163.71 | Eps 0.905 | Buf   3000 | Loss 0.0006
Saved best checkpoint at episode 10, eval mean 63.54
Episode   20 | TrainRet    92.96 | TrainAvg100    56.01 | EvalMean    76.66 | EvalMax   131.58 | Eps 0.810 | Buf   6000 | Loss 0.0049
Saved best checkpoint at episode 20, eval mean 76.66
Episode   30 | TrainRet   141.00 | TrainAvg100    66.43 | EvalMean   105.05 | EvalMax   194.56 | Eps 0.715 | Buf   9000 | Loss 0.0110
Saved best checkpoint at episode 30, eval mean 105.05
Episode   40 | TrainRet   195.16 | TrainAvg100    81.79 | EvalMean   157.35 | EvalMax   192.85 | Eps 0.620 | Buf  12000 | Loss 0.0226
Saved best checkpoint at episode 40, eval mean 157.35
Episode   50 | TrainRet    78.91 | TrainAvg100    86.79 | EvalMean   185.11 | EvalMax   275.27 | Eps 0.525 | Buf  15000 | Loss 0.0340
Saved best checkpoint at episode 50, eval mean 185.11
Episode   60 | TrainRet   208.33 | TrainAvg100   101.89 | Eval